# Two-Tower Model (TTN)

Build a PyTorch two-tower model on the Home & Kitchen interactions and the
extracted item features. See `README.md` in this folder for the design.

BPR-style pairwise ranking loss: `-log σ(score(u, i) - score(u, j))`.

In [13]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the user–item interactions

`data/Home_and_Kitchen_filtered.csv` is the interaction log: **one row per
review**, i.e. one row per (user, item, date) event. This is the table the
two-tower model is trained on — every other dataset in this notebook is joined
onto it.

**All 11 columns are loaded here** so the full contents are visible before
anything is thrown away. The next section decides what to keep.

`asin` and `reviewerID` are pinned to `str` so IDs with leading zeros
(e.g. `0560467893`) survive, and `low_memory=False` avoids the mixed-type
warning on `vote`.

In [14]:
from pathlib import Path

# data/ lives at the repo root, one level up from this ttn/ folder
DATA_DIR = Path("..") / "data"

df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    dtype={"asin": str, "reviewerID": str},
    low_memory=False,
)

print("df_reviews:", df_reviews.shape)
print("columns:", list(df_reviews.columns))
print(f"unique users: {df_reviews['reviewerID'].nunique():,} | "
      f"unique items: {df_reviews['asin'].nunique():,}")
df_reviews.head(5)

df_reviews: (6898955, 11)
columns: ['overall', 'verified', 'reviewTime', 'reviewerID', 'asin', 'reviewerName', 'summary', 'unixReviewTime', 'vote', 'style', 'image']
unique users: 777,242 | unique items: 189,172


,overall,verified,reviewTime,reviewerID,asin,reviewerName,summary,unixReviewTime,vote,style,image
0,5.00,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,Linda Fahner,Five Stars,1446681600,NaN,NaN,NaN
1,3.00,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,Harry Slaughter,Meh,1430956800,2,NaN,NaN
2,5.00,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,luckyg,Recommend,1390348800,NaN,{'Color:': ' Brushed Stainless'},NaN
3,1.00,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,Nickleen,Not keeping coffee hot for long enough,1383091200,NaN,{'Color:': ' Brushed Stainless'},NaN
4,1.00,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,Lacemaker427,Leaks like a waterfall when at an angle!,1379635200,NaN,{'Color:': ' Red'},NaN


### Duplicate rows

The check below counts rows that are identical across **all** columns. Because
every column is now loaded, this is the true count of the same review being
recorded twice: **231,139 rows**.

**Note — the duplicates are an artefact of not holding every variable.** A
"duplicate" only ever means *identical across the variables we happen to have*,
and that is never the full record. `Home_and_Kitchen_filtered.csv` is itself a
projection of the original review dump — `reviewText`, the review body, isn't in
it at all — so two rows that look identical here may well have differed in a
column dropped upstream. The narrower the projection, the more rows collapse
together:

| Columns compared | Rows reported as duplicates |
| --- | --- |
| all 11 CSV columns | **231,139** |
| the 7 columns this notebook loaded before (`overall`, `verified`, `reviewTime`, `reviewerID`, `asin`, `unixReviewTime`, `vote`) | **248,121** |

The extra ~17k are the same user buying two *variants* of an item on the same
day — genuinely different interactions that become indistinguishable once
`style` and `summary` are gone.

So if you decide to drop the duplicates, do it here — before the removal step
below takes those columns away.

In [10]:
df_reviews.duplicated().sum()

np.int64(231139)

### Variables to remove from the interaction table

Two different reasons to drop a column, and it matters which one applies:

**(a) Post-interaction — would leak at serving time.** These only exist *after*
the purchase happened. At recommendation time we don't have them, so a model
trained on them learns from information it will never see in production.

| Column | Why remove |
| --- | --- |
| `overall` | The star rating, given after the purchase. Training here is **implicit** — the interaction itself is the positive signal, so the rating is neither needed as a label nor usable as an input. |
| `vote` | Helpfulness votes, accumulated by other users *after* the review is posted. Later still than the rating. Also almost entirely NaN. |
| `summary` | The review's text, written after the purchase. Post-interaction, and unstructured text these towers don't consume. |
| `image` | Customer photos uploaded with the review. Post-interaction, and nearly always empty. |

**(b) No signal / redundant.** Available at interaction time, but carrying
nothing the model can use.

| Column | Why remove |
| --- | --- |
| `reviewerName` | A display name, not an identifier — not unique, and changeable. `reviewerID` already identifies the user. |
| `style` | The variant chosen (`{'Color:': ' Brushed Stainless'}`). Known at purchase time, but it describes a *variant* while the model recommends at `asin` level, and it's a free-form dict string with very high cardinality. |

**Kept:**

| Column | Why keep |
| --- | --- |
| `reviewerID`, `asin` | The interaction itself — the user and item towers' keys. |
| `unixReviewTime` | The timeline. Needed for the temporal train/test split, and it's the key the user features are aligned on. |
| `reviewTime` | Human-readable duplicate of `unixReviewTime`. Kept only for spot-checks; safe to drop for a leaner frame. |
| `verified` | A property of the *transaction*, not of the review content, so it's known at interaction time. Useful later as an interaction-quality filter — not as a tower input. |

⚠️ **One caveat about dropping `style` and `summary`:** they're what distinguishes
a genuine same-day purchase of two *different variants* of an item from the same
review being recorded twice — see the duplicate check above. If you de-duplicate
the log, do it **before** this cell; deduping afterwards collapses ~17k real
interactions.

In [15]:
# (a) post-interaction — known only after the purchase, would leak at serving time
POST_INTERACTION = ["overall", "vote", "summary", "image"]

# (b) no signal / redundant
NO_SIGNAL = ["reviewerName", "style"]

DROP_COLS = POST_INTERACTION + NO_SIGNAL

df_reviews = df_reviews.drop(columns=[c for c in DROP_COLS if c in df_reviews.columns])

print(f"dropped {len(DROP_COLS)}: {DROP_COLS}")
print(f"kept    {df_reviews.shape[1]}: {list(df_reviews.columns)}")
print("\ndf_reviews:", df_reviews.shape)
df_reviews.head(5)

dropped 6: ['overall', 'vote', 'summary', 'image', 'reviewerName', 'style']
kept    5: ['verified', 'reviewTime', 'reviewerID', 'asin', 'unixReviewTime']

df_reviews: (6898955, 5)


,verified,reviewTime,reviewerID,asin,unixReviewTime
0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600
1,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800
2,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800
3,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,1383091200
4,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,1379635200


## 2. Item features

`data/df_features.pkl` — one row per `asin` (~1.13M items) with the attributes
extracted by `feature_extraction_workflow/extract_features.py`: `cat_*`,
`brand`, the per-field columns (`Product_Type`, `Material`, `Color`, …), their
parsed `_numeric` / `_unit` / `_cleaned` measures, and `title_cleaned`.

It is loaded and **analysed on its own first**. The join onto the interactions
happens in §4, once the checks below have settled which columns are worth
carrying.

In [16]:
from pathlib import Path

# data/ lives at the repo root, one level up from this ttn/ folder
DATA_DIR = Path("..") / "data"

# --- Item features (one row per asin) ---
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")
print("df_features:", df_features.shape)
df_features.head(3)

df_features: (1134566, 83)


,category,tech1,description,title,tech2,brand,feature,rank,main_cat,price,asin,date,imageURL,imageURLHighRes,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6,title_cleaned,extracted_features_title,description_cleaned,extracted_features_description,feature_cleaned,extracted_features_feature,extracted_features,Bar_Pressure,Brand,Capacity,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,capacity_numeric,capacity_unit,capacity_volume_numeric,capacity_volume_unit,piece_count_numeric,piece_count_unit,thread_count_numeric,thread_count_unit,weight_numeric,weight_unit,bar_pressure_numeric,capacity_cups_numeric,density_weight_lb,pocket_depth_in,power_rating_w,stage_count_numeric,voltage_numeric,dimension_1,dimension_2,dimension_3,dimension_unit,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,density_weight_lb_cleaned,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,voltage_numeric_cleaned,thread_count_numeric_cleaned,weight_numeric_cleaned
0,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,['It was a time honored tradition among the ea...,You Are Special Today Red Plate [With Red Pen],NaN,Waechtersbach USA,[],"['>#39,665 in Kitchen & Dining (See Top 100 in...",Amazon Home,$37.00,0001487795,"October 8, 2006",[],[],Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates,special today red plate red pen,{'Color': 'red'},time honored tradition among early american fa...,{'Color': 'red'},,{},{'Color': 'red'},None,None,None,None,None,red,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"['Home & Kitchen', 'Home Dcor', 'Candles & Hol...",NaN,['VICKS INHALER relieves stuffy noses helps si...,Vicks Inhaler Relief for Cold Sinus Nasal Cong...,NaN,Vicks,[],"['>#1,763,185 in Home & Kitchen (See Top 100 i...",Amazon Home,$4.05,0002020300,NaN,[],[],Home & Kitchen,Home Dcor,Candles & Holders,Candles,None,None,vicks inhaler relief cold sinus nasal congesti...,{},vicks inhaler relief stuffy nose help sinus co...,{},,{},{},None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,"['16 oz squeeze bottle, 1 lb.']",Artistic Churchware Communion Cup Filler: RW525,NaN,Artistic Churchware,"['Religious Supply Center', 'RW-525', 'Communi...","['>#2,127,003 in Home & Kitchen (See Top 100 i...",Amazon Home,$12.48,0006564224,NaN,[],[],Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Wine & Champagne Glasses,None,artistic churchware communion cup filler rw525,{'Product_Type': 'cup'},16 oz squeeze bottle 1 lb,{'Capacity_Volume': '16 oz'},religious supply center rw-525 communion cup f...,{'Product_Type': 'cup'},"{'Product_Type': 'cup', 'Capacity_Volume': '16...",None,None,None,None,16 oz,None,None,None,None,None,None,None,None,None,None,cup,None,None,None,None,None,None,None,None,None,None,NaN,NaN,16.00,oz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Examine the item features — before joining

These columns are not ready to feed a tower as they stand, and this section is
the checklist that decides which of them survive.

**Why the analysis runs here, on `df_features`, and not after the join:** this
table is one row per `asin`. The merged frame repeats an item once per review,
so every distribution measured there is **popularity-weighted** — a `Color`
count on the merged frame describes reviews, not the catalog, and a heavily
reviewed item drags the numbers toward its own attributes. The keep/drop
decision is about item variables, so it belongs on the item table. It is also
1.13M rows instead of 6.9M.

(Coverage *over interactions* is a separate and also useful question — a field
present on few items can still cover many reviews. That one is worth checking
after the join in §4.)

What needs a look first:

- **Redundancy.** Several fields appear twice — the raw extracted string
  (`Weight`, `Capacity`, `Piece_Count`) and the parsed `_numeric` + `_unit`
  (+ range-cleaned `_cleaned`) version. Feeding both feeds the same information
  twice.
- **Mixed units.** Raw numbers aren't comparable until units are reconciled:
  `weight_numeric` mixes `lb` and `g`, `dimension_*` mixes `in` and `cm`. A
  model reading the number without the unit treats `1 lb` and `1 g` alike.
- **Cardinality.** `brand` has 27,909 distinct values and `Product_Type` 1,129,
  while `Color` has 67 and `Size` 20 — which drives embedding vs. one-hot vs.
  drop.
- **Coverage.** Some fields are populated on a small slice of items
  (`bar_pressure_numeric_cleaned`: ~9k). A near-empty column adds parameters
  and noise, not signal.
- **Text that is really numeric.** `Filter_Rating` holds `'5 micron'`,
  `'40 gallon'` — a number and a unit in one unparsed string.

Nothing is dropped until the checks below have all been run.

**First**, I need to standardize the numerical values of features and make them in the same units (e.g., inch -> cm). Once I do that, I may need to exclude some of the feature variables.

**Variables to Exclude** <br>
**Bar_Pressure:** a bar_pressure_numeric_cleaned variable is created that includes the necessary information <br>
**extracted_features:** every features is stored as a separate variables <br>
**Capacity_Cups:** capacity_cups_numeric_cleaned is the cleaned version <br>
**Density_weight:** density_weight_lb_cleaned is the cleaned version <br>
**Pocket_Depth:** pocket_depth_in_cleaned is the cleaned version <br>
**Power_Rating:** power_rating_w_cleaned is the cleaned version <br>
**Stage_Count:** stage_count_numeric_cleaned is the cleaned version <br>
**Voltage:** voltage_numeric_cleaned is the cleaned version <br>
**Thread_Count:** thread_count_numeric_cleaned and thread_count_unit are the cleaned versions <br>
**Weight:** weight_numeric_cleaned and weight_unit are the cleaned versions <br>
**Capacity:** capacity_numeric and capacity_unit are the cleaned versions <br>
**Capacity_Volume:** capacity_volume_numeric and capacity_volume_unit are the cleaned versions <br>
**Piece_Count:** piece_count_numeric and piece_count_unit are the cleaned versions

## **Variables to Check** <br>
**Categorical Variables** <br>
brand <br>
Brand <br>
Color <br>
Features <br>
Filter_Rating <br>
Material <br>
Part_Number <br>
Product_Type <br>
Scent <br>
Shape <br>
Shape_Style <br>
Size <br>
Sub_Type <br>
Theme <br>
<br>
<br>
**Variables consisting of two parts: unit and value** <br>

- capacity_numeric <br>
- capacity_unit <br>
<br>
- capacity_volume_numeric <br>
- capacity_volume_unit <br>
<br>
- piece_count_numeric <br>
- piece_count_unit <br>
<br>
- thread_count_numeric <br>
- thread_count_unit <br>
<br>
- weight_numeric <br>
- weight_unit <br>
<br>
<br>
**Variables with Range Filters** <br>
bar_pressure_numeric_cleaned <br>
capacity_cups_numeric_cleaned <br>
density_weight_lb_cleaned <br>
pocket_depth_in_cleaned <br>
power_rating_w_cleaned <br>
stage_count_numeric_cleaned <br>
voltage_numeric_cleaned <br>
<br>
**Dimension Variables** <br>
dimension_1 <br>
dimension_2 <br>
dimension_3 <br>
dimension_unit <br>
<br>
**Product Categories** <br>
cat_2 <br>
cat_3 <br>
cat_4 <br>


In [17]:
# Check categories: cat_2, cat_3, and cat_4
print(f'Unique cat_2 values: \n {df_features['cat_2'].unique()}')
print(f'Unique cat_3 values: \n {df_features['cat_3'].unique()}')
print(f'Unique cat_4 values: \n {df_features['cat_4'].unique()}')

Unique cat_2 values: 
 ['Kitchen & Dining' 'Home Dcor' 'Bath' 'Wall Art' 'Bedding' 'Furniture'
 "Kids' Home Store"]
Unique cat_3 values: 
 ['Dining & Entertaining' 'Candles & Holders' 'Bathroom Accessories'
 'Home Fragrance' 'Posters & Prints' "Kids' Bedding"
 'Storage & Organization' 'Home Dcor Accents' 'Blankets & Throws'
 'Kitchen Utensils & Gadgets' 'Travel & To-Go Drinkware' 'Clocks'
 'Picture Frames' 'Bakeware' 'Small Appliances' 'Quilts & Sets'
 'Bed Pillows & Positioners' 'Home Brewing & Wine Making'
 'Photo Albums & Accessories' 'Tapestries' "Kids' Furniture"
 'Comforters & Sets' 'Game & Recreation Room Furniture' "Kids' Room Dcor"
 'Duvets, Covers & Sets' 'Kitchen & Table Linens'
 'Area Rugs, Runners & Pads' 'Small Appliance Parts & Accessories'
 'Cutlery & Knife Accessories' 'Artificial Plants & Flowers'
 'Gift Baskets' 'Window Treatment Hardware' 'Cookware'
 'Home Office Furniture' 'Coffee, Tea & Espresso' 'Vases'
 'Bedroom Furniture' 'Decorative Pillows, Inserts & Covers'


### Cleaning `cat_4` with `category_taxonomy.json`

The check above shows ~920 distinct `cat_4` values, and most of them are not
categories at all — they are **product bullets that leaked into the category
path**: `10" high`, `Imported`, `Material:Paper+Plastic`,
`</span></span></span>`. `ensure_cat_columns` parses the source `category`
list positionally, so when an item's list has bullets appended after the real
taxonomy path, they land in `cat_4` and below. Note that `cat_3` is clean:
`filter_by_cat_3` keeps only rows whose `cat_3` matches a key in
`master_metadata.json`. Nothing validates level 4 — hence the mess.

**How the valid list was decided.** For every `cat_2 / cat_3 / cat_4` path we
counted the **number of unique items (`asin`) carrying it**. That count is the
signal: a real category is shared by many products, while a bullet belongs to
exactly one listing and shows up with a count of 1. Values below ~20 items were
treated as invalid, and the ones above the threshold were eyeballed as well —
which caught three that frequency alone would have kept (`Canvas` at 26,
`High quality poster paper material` at 24, `</span></span></span>` at 46,
all under Posters & Prints).

Those decisions are stored in **`data/category_taxonomy.json`** as the surviving
hierarchy: `cat_2 → cat_3 → [valid cat_4 values]`.

**The rule applied below:**

| Case | Result |
| --- | --- |
| `cat_4` is in its `cat_3`'s list | kept unchanged |
| `cat_4` is absent from the source path (`NaN`) | `Missing` |
| anything else (bullet text, or too few items) | `<cat_3>_Other` |

Backing off to the **parent** rather than one global `Other` matters: a shared
bucket would force the embedding for `Other` to average a poster bullet and a
kitchen bullet into one meaningless vector. `Bakeware_Other` instead sits
naturally near `Bakeware`, and says something true — "a bakeware item with no
usable subcategory".

Result: **921 → 451 distinct values**, and it costs almost nothing —
about **0.1% of items** end up in an `_Other` bucket.

In [ ]:
import json

TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

# cat_3 -> the set of cat_4 values that survived the review
valid_cat_4 = {c3: set(vals) for c2 in taxonomy for c3, vals in taxonomy[c2].items()}
print(f"taxonomy: {len(taxonomy)} cat_2 | {len(valid_cat_4)} cat_3 | "
      f"{sum(len(v) for v in valid_cat_4.values())} valid cat_4 slots")

MISSING = "Missing"
OTHER_SUFFIX = "_Other"

cat_3 = df_features["cat_3"].astype(str)
cat_4 = df_features["cat_4"].fillna(MISSING).astype(str)

# A value is kept only if it is valid *under its own parent* — the same label
# can be real in one branch and junk in another.
valid_pairs = {(c3, v) for c3, vals in valid_cat_4.items() for v in vals}
keep = pd.Series(list(zip(cat_3, cat_4)), index=df_features.index).isin(valid_pairs)

df_features["cat_4_clean"] = np.where(keep, cat_4, cat_3 + OTHER_SUFFIX)

n_before = df_features["cat_4"].nunique(dropna=False)
n_after = df_features["cat_4_clean"].nunique()
n_folded = int((~keep).sum())
print(f"\ndistinct cat_4 : {n_before:,} -> {n_after:,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features) * 100:.2f}% of the catalog)")
df_features[["asin", "cat_2", "cat_3", "cat_4", "cat_4_clean"]].head(5)

In [ ]:
# Before / after on the two branches that motivated this: one where the junk was
# a long tail, and one where it was nearly the whole branch.
for c3 in ("Kitchen Utensils & Gadgets", "Posters & Prints"):
    sub = df_features[df_features["cat_3"] == c3]
    print(f"\n=== {c3} ===")
    print(f"raw cat_4: {sub['cat_4'].nunique(dropna=False)} distinct  ->  "
          f"clean: {sub['cat_4_clean'].nunique()} distinct")
    print(sub["cat_4_clean"].value_counts().to_string())

Check **categorical** variables

In [ ]:
for i in ('brand', 'Color', 'Features', 'Filter_Rating', 'Material', 'Part_Number',
          'Product_Type', 'Scent', 'Shape', 'Shape_Style', 'Size', 'Sub_Type', 'Theme'):
    print(f'Unique number of {i}: {df_features[i].nunique()}')
    print(f'First 10 unique {i}: \n {df_features[i].unique()[:10]} \n')

Check for numeric **cleaned variables**

In [ ]:
for i in ('bar_pressure_numeric_cleaned', 'capacity_cups_numeric_cleaned',
          'density_weight_lb_cleaned', 'pocket_depth_in_cleaned',
          'power_rating_w_cleaned', 'stage_count_numeric_cleaned', 'voltage_numeric_cleaned'):

    print(f'Summary of {i}: \n {df_features[i].describe()}')

In [ ]:
df_features[df_features['piece_count_unit'].isna() == False][['asin', 'piece_count_numeric', 'piece_count_unit']].head(10)
# df_features['piece_count_unit'].unique()

In [ ]:
df_features[df_features['weight_numeric_cleaned'].isna() == False]['weight_unit'].unique()

## 4. Join the item features onto the interactions

With the examination above settled, attach the item columns to the interaction
table on `asin`. Left join, so no interaction is dropped: items with no
extracted features keep their row with NaN item columns (~16% of rows —
those asins either aren't in the item metadata, or were dropped during
extraction because their `cat_3` has no schema).

The cell below currently attaches **every** field and measure column. Once §3
has produced a keep-list, narrow `keep_cols` to it — that is the one place the
exclusion decision needs to be applied.

In [ ]:
# --- Connect the two on `asin` ---
# Attach ALL extracted feature variables plus their parsed measures.

# 1. Every extracted feature field column (the keys present in the dicts):
#    Product_Type, Material, Color, Weight, Dimensions, Brand, Theme, ...
field_cols = sorted({
    k for d in df_features["extracted_features"]
    if isinstance(d, dict) for k in d
})

# 2. Their parsed measures: numeric value + unit (+ dimension_1/2/3).
#    Use the range-cleaned numeric variant wherever one exists.
measure_cols = [
    c for c in df_features.columns
    if c.endswith("_numeric") or c.endswith("_unit")
    or c.startswith("dimension_")
    or c in ("density_weight_lb", "pocket_depth_in", "power_rating_w")
]
measure_cols = sorted({
    f"{c}_cleaned" if f"{c}_cleaned" in df_features.columns else c
    for c in measure_cols
})

# 3. Context columns to carry along (asin is the join key).
# cat_4_clean (from category_taxonomy.json) replaces the raw cat_4 here —
# the ~920 raw values are mostly bullet text, see section 3.
context_cols = ["asin", "cat_2", "cat_3", "cat_4_clean", "brand", "extracted_features"]

keep_cols = list(dict.fromkeys(context_cols + field_cols + measure_cols))
keep_cols = [c for c in keep_cols if c in df_features.columns]

df = df_reviews.merge(
    df_features[keep_cols],
    on="asin",
    how="left",
    validate="many_to_one",   # many reviews -> one item row
    indicator=True,
)
n_unmatched = (df["_merge"] == "left_only").sum()
df = df.drop(columns="_merge")

print(f"merged: {df.shape}  ({len(keep_cols)} item cols attached)")
print(f"  feature fields : {len(field_cols)}")
print(f"  measure cols   : {len(measure_cols)}")
print(f"unique users: {df['reviewerID'].nunique():,} | "
      f"unique items: {df['asin'].nunique():,}")
print(f"reviews with no matching item features: {n_unmatched:,}")
df.head(5)

## 5. User features

`data/df_user_features.pkl` — built by
`feature_extraction_workflow/extract_features_user.py`. **One row per purchase
event** with 22 features, each computed only from that user's purchases on
**strictly earlier days**, so no row can see its own or any later interaction.
See that module's section in `feature_extraction_workflow/README.md`.

### Why this is a positional concat, not a merge

The natural key is (user, item, date) — joining on `reviewerID` alone would be a
real leak, attaching every event's feature vector, including future ones, to
every row of that user.

But that triple **isn't unique in this log**: 501,704 rows share a
`(reviewerID, asin, unixReviewTime)` triple with at least one other row, so a
key merge is many-to-many and square-joins those groups into **+572,198 phantom
rows (+8.3%)**.

The pickle was generated from this exact CSV, in file order, dropping nothing —
so row *i* of the pickle already is row *i* of the log. Aligning by position
gives identical semantics with zero fanout.

The asserts below **verify** that alignment element-wise instead of assuming it.
They also confirm the row order survived the item join in §4. If the pickle is
ever regenerated from a different or reordered log, this fails loudly rather
than silently pairing the wrong user's history to a row.

**Expect ~23.5% of rows to have all-NaN user features.** That's a user's first
purchase (and anything on that same first day) — there is no prior history to
summarize. Intended, not a defect.

In [ ]:
df_user_features = pd.read_pickle(DATA_DIR / "df_user_features.pkl")
print("df_user_features:", df_user_features.shape)
display(df_user_features.head(3))

# --- Verify positional alignment before relying on it ---
assert len(df_user_features) == len(df), (
    f"row count mismatch: user features {len(df_user_features):,} vs "
    f"interactions {len(df):,}"
)
for key in ("reviewerID", "asin", "unixReviewTime"):
    n_bad = int((df_user_features[key].to_numpy() != df[key].to_numpy()).sum())
    assert n_bad == 0, f"row order mismatch on {key}: {n_bad:,} rows differ"
print("alignment verified: reviewerID / asin / unixReviewTime match row-for-row\n")

# Attach the 22 feature columns (the key columns are already in df)
USER_FEATURE_COLS = [
    c for c in df_user_features.columns
    if c not in ("reviewerID", "asin", "reviewTime", "unixReviewTime")
]
df = pd.concat(
    [df.reset_index(drop=True),
     df_user_features[USER_FEATURE_COLS].reset_index(drop=True)],
    axis=1,
)

n_cold = int(df["prior_purchase_count"].isna().sum())
print(f"attached {len(USER_FEATURE_COLS)} user feature columns -> df {df.shape}")
print(f"cold-start rows (empty history, all user features NaN): "
      f"{n_cold:,} ({n_cold / len(df) * 100:.1f}%)")
df.head(5)